# 01. Phase 1 — ESMFold 베이스라인
tau K18, α-synuclein, Aβ42에 대해 ESMFold pseudo-ensemble 생성

In [ ]:
import os
os.chdir('/content/brain_idp_flow')

import numpy as np
import torch
from brain_idp_flow.targets import load_targets
from brain_idp_flow.baseline.esmfold_infer import generate_esmfold_ensemble

targets = load_targets('configs/targets.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ESMFold로 각 타깃의 WT pseudo-ensemble 생성 (50개씩)
os.makedirs('samples/baseline', exist_ok=True)

for tid, target in targets.items():
    print(f'\n=== {target.name} (L={target.length}) ===')
    coords, plddt = generate_esmfold_ensemble(
        target.sequence, n_samples=50, device=device
    )
    np.savez(
        f'samples/baseline/{tid}.npz',
        coords=coords, plddt=plddt
    )
    print(f'  coords: {coords.shape}')
    print(f'  pLDDT mean: {plddt.mean():.1f} (low = ESMFold struggles with IDP)')
    print(f'  pLDDT per-sample range: [{plddt.mean(axis=1).min():.1f}, {plddt.mean(axis=1).max():.1f}]')

In [ ]:
# 시각화: Rg 분포 (baseline vs PED reference)
from brain_idp_flow.data.ped_loader import load_ped_or_fallback
from brain_idp_flow.eval.plots import plot_rg_comparison, plot_3d_traces

for tid, target in targets.items():
    data = np.load(f'samples/baseline/{tid}.npz')
    ref = load_ped_or_fallback(target.ped_id, target.length)
    
    fig = plot_rg_comparison(
        {'PED reference': ref, 'ESMFold baseline': data['coords']},
        title=f'{target.name} — Rg (Phase 1 Baseline)',
    )
    fig.show()
    
    fig2 = plot_3d_traces(
        data['coords'], n_traces=10,
        title=f'{target.name} — ESMFold Cα Traces'
    )
    fig2.show()

In [ ]:
# 정량 비교
from brain_idp_flow.eval.compare import compare_ensembles

for tid, target in targets.items():
    data = np.load(f'samples/baseline/{tid}.npz')
    ref = load_ped_or_fallback(target.ped_id, target.length)
    metrics = compare_ensembles(data['coords'], ref)
    print(f'{tid}: {metrics.summary()}')